# Object-Level Paired Test (Faster R-CNN, SSL): Does L Carry Unique Information vs R?

Pairs at the **object** level, not the image level. RGB and LGB share identical images and identical
label files, so every ground-truth (GT) box has one score from the RGB model and one from the LGB model
for the **same physical object**.

Two comparisons, testing two different claims:

1. **Only R vs Only L**: is a single channel (R alone / L alone) informative on its own?
2. **Mask R vs Mask L**: with G and B still present, does removing R hurt less than removing L?
   If yes, G/B can substitute for R but not for L, i.e. **L carries information G/B do not already have**.

**DIP reminder:** an image is 3 matrices (planes). Masking = setting one plane to zero.
Example pixel (200, 100, 50): *Mask R* -> (0, 100, 50) | *Only R* -> (200, 0, 0).

In [1]:
import os, json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from scipy import stats
from tqdm.auto import tqdm
from torchvision.ops import box_iou
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [2]:
DATASET = "SSL"

BASE_DIR = Path(r"C:\Users\DELL\Desktop\c")
RUNS_DIR = BASE_DIR / "runs_frcnn_neutral"
IMBALANCE_MODE = "ce"            # must match the folder names of your training runs

# same class order as the training notebook / YOLO yaml
CLASS_NAMES = ['airplane', 'boat', 'car', 'ship']
NC = len(CLASS_NAMES)
NUM_CLASSES = NC + 1             # +1 = background (label 0)

DATASETS = {c: BASE_DIR / f"{DATASET}_{c}" for c in ("RGB", "LGB")}
CKPTS    = {c: RUNS_DIR / f"{DATASET}_{c}_FasterRCNN_{IMBALANCE_MODE}" / "best.pth" for c in ("RGB", "LGB")}

PRED_CONF = 0.001      # keep even very weak detections, so a score is not forced to 0 by a threshold
IOU_THR   = 0.5
FILL_MODE = "zero"     # "zero" or "mean"
CH        = 0          # PIL/torch order: index 0 = first stored channel = R (RGB folder) / L (LGB folder)
MIN_SIZE, MAX_SIZE = 800, 1333      # same as training
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

CONFIGS = {
    "Only": "keep_only",   # Only R  vs  Only L
    "Mask": "zero_ch",     # Mask R  vs  Mask L
}

for c in ("RGB", "LGB"):
    print(c, "| data:", DATASETS[c], "| ckpt exists:", CKPTS[c].exists())

RGB | data: C:\Users\DELL\Desktop\c\SSL_RGB | ckpt exists: True
LGB | data: C:\Users\DELL\Desktop\c\SSL_LGB | ckpt exists: True


In [3]:
def find_split_dir(root, split):
    for p in Path(root).iterdir():            # case-insensitive: 'test' or 'TEST'
        if p.is_dir() and p.name.lower() == split.lower():
            return p
    raise FileNotFoundError(f"No '{split}' folder inside {root}")

def parse_yolo_file(path):
    rows = []
    if Path(path).exists():
        for line in Path(path).read_text().splitlines():
            parts = line.split()
            if len(parts) >= 5:
                rows.append([float(v) for v in parts[:5]])
    return np.array(rows, dtype=np.float32).reshape(-1, 5)

def load_image(path):
    img = Image.open(path).convert("RGB")          # in LGB folders the 3 stored channels are simply (L,G,B)
    return torch.from_numpy(np.asarray(img).copy()).permute(2, 0, 1).float() / 255.0, img.size

def load_gt(lbl_path, W, H):
    # YOLO (class, cx, cy, w, h) normalised  ->  pixel xyxy + label (1..N, 0 = background)
    a = parse_yolo_file(lbl_path)
    cls = a[:, 0].astype(np.int64) + 1
    cx, cy, bw, bh = a[:, 1]*W, a[:, 2]*H, a[:, 3]*W, a[:, 4]*H
    boxes = np.stack([cx - bw/2, cy - bh/2, cx + bw/2, cy + bh/2], axis=1).astype(np.float32)
    boxes[:, [0, 2]] = boxes[:, [0, 2]].clip(0, W)
    boxes[:, [1, 3]] = boxes[:, [1, 3]].clip(0, H)
    keep = ((boxes[:, 2] - boxes[:, 0]) >= 1) & ((boxes[:, 3] - boxes[:, 1]) >= 1)   # same filter as training
    return torch.as_tensor(boxes[keep]).reshape(-1, 4), torch.as_tensor(cls[keep], dtype=torch.int64)

def apply_mask(img, ch, kind, fill="zero"):
    out = img.clone()                               # img: 3 x H x W in [0,1]
    def f(c):
        out[c] = 0.0 if fill == "zero" else img[c].mean()
    if kind == "zero_ch":
        f(ch)
    else:                                           # keep_only
        for c in range(3):
            if c != ch:
                f(c)
    return out

def score_per_gt(pred, gb, gl, iou_thr):
    # One score per GT box, SAME order as gb. This order preservation is what makes object-level pairing valid.
    # Greedy matching: highest-score prediction first, same class, IoU >= thr, each GT claimed once. Missed -> 0.
    pb, ps, pl = pred["boxes"].cpu(), pred["scores"].cpu(), pred["labels"].cpu()
    scores = np.zeros(len(gb), dtype=np.float32)
    if len(pb) == 0 or len(gb) == 0:
        return scores
    order = ps.argsort(descending=True)
    pb, ps, pl = pb[order], ps[order], pl[order]
    ious = box_iou(pb, gb)
    used = torch.zeros(len(gb), dtype=torch.bool)
    for i in range(len(pb)):
        cand = ious[i].clone()
        cand[(gl != pl[i]) | used] = -1             # wrong class or already claimed
        j = int(cand.argmax())
        if cand[j] >= iou_thr:
            used[j] = True
            scores[j] = float(ps[i])
    return scores

def build_model():
    # same architecture as training; the COCO download is not needed because best.pth overwrites all weights
    model = fasterrcnn_resnet50_fpn_v2(weights=None, weights_backbone=None,
                                        min_size=MIN_SIZE, max_size=MAX_SIZE,
                                        box_score_thresh=PRED_CONF, box_detections_per_img=300)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES)
    return model

## Build the shared image list (RGB and LGB must share identical filenames)

In [4]:
def list_images(root):
    d = find_split_dir(root, "test") / "images"
    return d, sorted(p.name for p in d.iterdir() if p.suffix.lower() in IMG_EXTS)

img_dir, lbl_dir = {}, {}
names = {}
for c in ("RGB", "LGB"):
    img_dir[c], names[c] = list_images(DATASETS[c])
    lbl_dir[c] = find_split_dir(DATASETS[c], "test") / "labels"
assert names["RGB"] == names["LGB"], "RGB and LGB test folders do not share identical filenames -> pairing is invalid."

img_list = [n for n in names["RGB"] if (lbl_dir["RGB"] / (Path(n).stem + ".txt")).exists()]
print(f"{len(img_list)} shared test images")

127 shared test images


## Main loop

For each model (one at a time, to save GPU memory) and each image: load GT **once** from the RGB label file
(identical for both models), apply the channel mask, run the model, and store one score per GT object under
the explicit key `(image, box_idx)`. Explicit keys mean a skipped or corrupt image cannot silently misalign the pairing.

In [5]:
@torch.no_grad()
def collect_scores(color):
    assert CKPTS[color].exists(), f"checkpoint not found: {CKPTS[color]}"
    model = build_model()
    model.load_state_dict(torch.load(CKPTS[color], map_location="cpu", weights_only=True))
    model.to(device).eval()
    out = {cfg: {} for cfg in CONFIGS}               # out[cfg][(image, box_idx)] = (class_label, score)
    for name in tqdm(img_list, desc=color):
        img, (W, H) = load_image(img_dir[color] / name)
        gb, gl = load_gt(lbl_dir["RGB"] / (Path(name).stem + ".txt"), W, H)     # GT always from the RGB label file
        if len(gb) == 0:
            continue
        for cfg, kind in CONFIGS.items():
            x = apply_mask(img, CH, kind, FILL_MODE).to(device)
            pred = model([x])[0]
            s = score_per_gt(pred, gb, gl, IOU_THR)
            for j, sc in enumerate(s):
                out[cfg][(name, j)] = (int(gl[j]), float(sc))
    del model
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return out

raw = {c: collect_scores(c) for c in ("RGB", "LGB")}

paired_records = {cfg: [] for cfg in CONFIGS}
for cfg in CONFIGS:
    for key in sorted(set(raw["RGB"][cfg]) & set(raw["LGB"][cfg])):
        cls_r, sR = raw["RGB"][cfg][key]
        cls_l, sL = raw["LGB"][cfg][key]
        assert cls_r == cls_l
        paired_records[cfg].append({"image": key[0], "box_idx": key[1], "class": cls_r, "score_R": sR, "score_L": sL})
    print(f"{cfg}: {len(paired_records[cfg])} object pairs")

RGB:   0%|          | 0/127 [00:00<?, ?it/s]

LGB:   0%|          | 0/127 [00:00<?, ?it/s]

Only: 2075 object pairs
Mask: 2075 object pairs


## Paired significance tests

Run separately for `Only` (Only R vs Only L) and `Mask` (Mask R vs Mask L), pooled across classes and per class.

For each object, `d = score_R - score_L`:
- **Paired t-test**: `t = mean(d) / (std(d)/sqrt(n))`. Is the mean difference different from 0?
- **Wilcoxon**: rank-based, no normality assumption (scores are not normal, many are 0). It ignores pairs with `d = 0` (both models missed the object).
- **Cohen's d**: `mean(d)/std(d)`, the effect size (about 0.2 small, 0.5 medium, 0.8 large).
- Tiny example: `d = [0.1, 0.3, 0.2]` -> mean 0.2, std 0.1, t = 3.46, Cohen's d = 2.0.

In [6]:
def run_tests(records, label):
    r = np.array([x["score_R"] for x in records]); l = np.array([x["score_L"] for x in records])
    diffs = r - l
    t_stat, p_t = stats.ttest_rel(r, l)
    try:
        w_stat, p_w = stats.wilcoxon(r, l)
    except ValueError:                                   # all differences are zero
        w_stat, p_w = float("nan"), float("nan")
    sd = diffs.std(ddof=1)
    d = diffs.mean() / sd if sd > 0 else float("nan")
    print(f"\n{label}  (N={len(records)} objects)")
    print(f"  mean R={r.mean():.4f}  mean L={l.mean():.4f}  diff(R-L)={diffs.mean():+.4f}")
    print(f"  paired t-test:  t={t_stat:.4f}, p={p_t:.6f} {'(SIGNIFICANT)' if p_t < 0.05 else '(not significant)'}")
    print(f"  Wilcoxon:       W={w_stat:.4f}, p={p_w:.6f} {'(SIGNIFICANT)' if p_w < 0.05 else '(not significant)'}")
    print(f"  Cohen's d:      {d:.4f}")
    return [label, len(records), r.mean(), l.mean(), diffs.mean(), t_stat, p_t, w_stat, p_w, d]

summary = []
summary.append(run_tests(paired_records["Only"], "Only R vs Only L (pooled, all classes)"))
summary.append(run_tests(paired_records["Mask"], "Mask R vs Mask L (pooled, all classes)"))

print("\n--- Per-class breakdown ---")
for cfg in CONFIGS:
    print(f"\n=== {cfg} ===")
    for c in range(1, NC + 1):                           # labels are 1..N (0 = background)
        recs = [x for x in paired_records[cfg] if x["class"] == c]
        if len(recs) < 5:
            continue                                     # too few pairs for a meaningful test
        summary.append(run_tests(recs, f"{cfg}: {CLASS_NAMES[c-1]}"))

cols = ["test", "N", "mean_R", "mean_L", "diff_R-L", "t", "p_t", "W", "p_wilcoxon", "cohens_d"]
summary_df = pd.DataFrame(summary, columns=cols)
out_csv = RUNS_DIR / f"{DATASET}_FasterRCNN_object_paired_R_vs_L.csv"
summary_df.to_csv(out_csv, index=False)
print("\nsaved:", out_csv)


Only R vs Only L (pooled, all classes)  (N=2075 objects)
  mean R=0.0733  mean L=0.2748  diff(R-L)=-0.2014
  paired t-test:  t=-27.4424, p=0.000000 (SIGNIFICANT)
  Wilcoxon:       W=40646.0000, p=0.000000 (SIGNIFICANT)
  Cohen's d:      -0.6024

Mask R vs Mask L (pooled, all classes)  (N=2075 objects)
  mean R=0.6351  mean L=0.6966  diff(R-L)=-0.0615
  paired t-test:  t=-8.2705, p=0.000000 (SIGNIFICANT)
  Wilcoxon:       W=737522.0000, p=0.000000 (SIGNIFICANT)
  Cohen's d:      -0.1816

--- Per-class breakdown ---

=== Only ===

Only: airplane  (N=553 objects)
  mean R=0.0040  mean L=0.1554  diff(R-L)=-0.1515
  paired t-test:  t=-10.6696, p=0.000000 (SIGNIFICANT)
  Wilcoxon:       W=0.0000, p=0.000000 (SIGNIFICANT)
  Cohen's d:      -0.4537

Only: boat  (N=334 objects)
  mean R=0.0665  mean L=0.2820  diff(R-L)=-0.2155
  paired t-test:  t=-12.3039, p=0.000000 (SIGNIFICANT)
  Wilcoxon:       W=1962.0000, p=0.000000 (SIGNIFICANT)
  Cohen's d:      -0.6732

Only: car  (N=798 objects)
  me

## How to read this

- **Only R vs Only L, p < 0.05 with negative mean diff (L > R)**: L is more informative than R *on its own*.
  A skeptic could still say L just captures brightness/structure that G or B already provide.
- **Mask R vs Mask L, p < 0.05 with Mask-L scores lower (R compensates better than L)**: the stronger claim.
  G and B are present in both cases; if removing L hurts more even then, G/B cannot substitute for what L provides,
  which is direct evidence that L carries information distinct from G/B.
- Report both. If only the `Only` test is significant, temper the wording to "L is a strong single-channel proxy"
  rather than "L provides unique, non-redundant information".
- Many objects score 0 in both models (especially under `Only`), so trust Wilcoxon and Cohen's d over the t-test alone.